# Gaussian Processes and Kernel Choice

This notebook covers the theoretical foundations of the GP surrogate used in GP-WLS:
what a Gaussian Process is, how conditioning gives us a posterior, and why the Matérn kernel is the right choice over the more common RBF.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
matplotlib.rcParams = plt.rcParams
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

## 1.  What is a Gaussian Process?

A **Gaussian Process (GP)** is a probability distribution over *functions*. Just as a multivariate Gaussian defines a distribution over vectors, a GP defines a distribution over functions $f : \mathbb{R}^d \to \mathbb{R}$.

Formally, $f \sim \mathcal{GP}(m, k)$ means: for any finite collection of input points $x_1, \ldots, x_n$, the joint distribution of the function values is Gaussian:

$$\begin{pmatrix} f(x_1) \\ \vdots \\ f(x_n) \end{pmatrix} \sim \mathcal{N}(\mathbf{m}, K)$$

where
- $m_i = m(x_i)$ is the **mean function** (usually set to zero)
- $K_{ij} = k(x_i, x_j)$ is the **kernel matrix** --> the covariance between function values at $x_i$ and $x_j$

The kernel $k$ encodes our prior belief: points that are close in input space should have similar (correlated) outputs.

## 2.  Joint Gaussian distribution and conditioning

The power of GPs comes from the **conditioning rule** for multivariate Gaussians.

Suppose we have observed data $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$ where $y_i = f(x_i) + \varepsilon_i$ and $\varepsilon_i \sim \mathcal{N}(0, \sigma_n^2)$. We want to predict at a new point $x^*$.

The **joint prior** over observed values and the prediction is:

$$\begin{pmatrix} \mathbf{y} \\ f(x^*) \end{pmatrix} \sim \mathcal{N}\left( \mathbf{0},\ \begin{pmatrix} K(X,X) + \sigma_n^2 I & K(X, x^*) \\ K(x^*, X) & K(x^*, x^*) \end{pmatrix} \right)$$

Conditioning on the observations $\mathbf{y}$ gives the **posterior**:

$$f(x^*) \mid \mathbf{y} \sim \mathcal{N}(\mu(x^*),\ \sigma^2(x^*))$$

with

$$\mu(x^*) = K(x^*, X)\, [K(X,X) + \sigma_n^2 I]^{-1}\, \mathbf{y}$$

$$\sigma^2(x^*) = K(x^*, x^*) - K(x^*, X)\, [K(X,X) + \sigma_n^2 I]^{-1}\, K(X, x^*)$$

The posterior mean $\mu(x^*)$ is our best prediction. The posterior variance $\sigma^2(x^*)$ quantifies uncertainty. It is zero at observed points and grows as we move away from them.

### Cholesky decomposition

The matrix inverse $[K + \sigma_n^2 I]^{-1}$ appears in both expressions. Direct inversion is numerically unstable and costs $O(n^3)$. Instead we compute the **Cholesky factorization**:

$$K + \sigma_n^2 I = L L^\top$$

where $L$ is lower triangular. Solving $L L^\top \alpha = \mathbf{y}$ via two triangular solves costs $O(n^2)$ after the $O(n^3)$ factorization. Therefore, it is more stable and efficient.

## 3.  The kernel function

The kernel $k(x, x')$ measures **similarity** between two points. It fully determines the prior over functions:
- If $k(x, x')$ is large when $x \approx x'$ and decays quickly, the GP prior prefers **rough** functions.
- If $k(x, x')$ decays slowly, the GP prior prefers **smooth** functions.

A kernel must be **positive semi-definite**: for any set of points, the matrix $K$ must be PSD. This ensures the joint distribution is a valid Gaussian.

Most stationary kernels depend only on the distance $r = \|x - x'\|$ and have the form:

$$k(x, x') = \sigma_f^2 \cdot \phi(r / \ell)$$

with two hyperparameters:
- $\ell$ — **length scale**: how quickly similarity decays with distance (frequency of the function)
- $\sigma_f^2$ — **output variance**: the overall scale of the function (amplitude of the function)

## 4.  RBF kernel

The **Radial Basis Function (RBF)** kernel, also called the Squared Exponential:

$$k_{\text{RBF}}(x, x') = \sigma_f^2 \exp\left(-\frac{r^2}{2\ell^2}\right)$$

Functions sampled from a GP with RBF kernel are **infinitely differentiable**. Analytic everywhere, no roughness at any scale. This is mathematically elegant but physically unrealistic for many problems.

The RBF kernel corresponds to the limit $\nu \to \infty$ of the Matern kernel.

## 5.  Matern kernel 

The **Matern kernel** with smoothness parameter $\nu > 0$:

$$k_{\nu}(r) = \sigma_f^2 \cdot \frac{2^{1-\nu}}{\Gamma(\nu)} \left(\frac{\sqrt{2\nu}\, r}{\ell}\right)^\nu K_\nu\!\left(\frac{\sqrt{2\nu}\, r}{\ell}\right)$$

where $K_\nu$ is the modified Bessel function. Functions sampled from this kernel are exactly $\lceil \nu \rceil - 1$ times differentiable.

For half-integer values of $\nu$, the expression simplifies:

| $\nu$ | Differentiability | Closed form |
|---|---|---|
| 1/2 | 0 (continuous only) | $\sigma_f^2\, e^{-r/\ell}$ |
| 3/2 | 1 | $\sigma_f^2\,(1 + \frac{\sqrt{3}r}{\ell})\, e^{-\sqrt{3}r/\ell}$ |
| 5/2 | 2 | $\sigma_f^2\,(1 + \frac{\sqrt{5}r}{\ell} + \frac{5r^2}{3\ell^2})\, e^{-\sqrt{5}r/\ell}$ |
| $\infty$ | $\infty$ | RBF |

### Why ν = 2.5 for this project

GP-WLS performs **gradient ascent** on the posterior mean $\mu(x)$ via PyTorch autograd. For this gradient to exist and be stable, $\mu(x)$ must be at least once differentiable, which requires $\nu \geq 3/2$.

We use $\nu = 5/2$ because:
1. The posterior mean is **twice differentiable** --> reliable, smooth gradients for ascent
2. It does not assume infinite smoothness like RBF --> better suited to functions that are smooth but not analytic
3. It is the standard choice for optimization on physical and engineering problems in the literature (e.g., Snoek et al., 2012; Rasmussen & Williams, 2006)

The RBF kernel, while common, would produce overconfident predictions between observations by assuming far more smoothness than is warranted.

In [ ]:
# Visualise how similarity decays with distance for each kernel

r = torch.linspace(0, 4, 300)
l = 1.0

rbf      = torch.exp(-r**2 / (2 * l**2))
m05      = torch.exp(-r / l)
m15      = (1 + (3**0.5)*r/l) * torch.exp(-(3**0.5)*r/l)
m25      = (1 + (5**0.5)*r/l + 5*r**2/(3*l**2)) * torch.exp(-(5**0.5)*r/l)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r, rbf, color="gray",    linewidth=2, linestyle="--", label="RBF ($\\nu \\to \\infty$)")
ax.plot(r, m05, color="#d62728", linewidth=2, label="Matérn $\\nu=1/2$")
ax.plot(r, m15, color="#ff7f0e", linewidth=2, label="Matérn $\\nu=3/2$")
ax.plot(r, m25, color="#1f77b4", linewidth=2, label="Matérn $\\nu=5/2$ (used)")

ax.set_xlabel("Distance  $r = \\|x - x'\\|$", fontsize=12)
ax.set_ylabel("Normalised kernel value", fontsize=12)
ax.set_title("Similarity decay: RBF vs Matérn", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../experiments/results/kernel_decay.png", dpi=150)
plt.show()

RBF decays very slowly --> distant points remain correlated, enforcing global smoothness. Matern ν=5/2 (our choice) decays faster and is strictly local: only nearby points influence the prediction.

## 6.  Hyperparameter learning via Log Marginal Likelihood

The kernel hyperparameters $\theta = (\ell, \sigma_f^2)$ are not fixed -> they are learned from the observed data $\mathcal{D}$ by maximising the **Log Marginal Likelihood (LML)**:

$$\log p(\mathbf{y} \mid X, \theta) = -\frac{1}{2}\mathbf{y}^\top(K_\theta + \sigma_n^2 I)^{-1}\mathbf{y} - \frac{1}{2}\log|K_\theta + \sigma_n^2 I| - \frac{n}{2}\log 2\pi$$

The three terms have clear interpretations:
- **Data fit** $-\frac{1}{2}\mathbf{y}^\top(K+\sigma_n^2 I)^{-1}\mathbf{y}$: how well the model explains the observations
- **Complexity penalty** $-\frac{1}{2}\log|K+\sigma_n^2 I|$: penalises overly complex models (large kernel matrices)
- **Normalisation** $-\frac{n}{2}\log 2\pi$: constant

The LML automatically balances fit and complexity. Maximisation is done via **L-BFGS** with strong Wolfe conditions (from `torch.optim.LBFGS`).

In the implementation, $\ell$ and $\sigma_f$ are stored in **log-space** (`log_length_scale`, `log_output_variance`) so that gradient-based optimization is unconstrained --> never need to enforce positivity explicitly:

$$\ell = e^{\log \ell}, \quad \sigma_f = e^{\log \sigma_f}$$

The noise variance $\sigma_n^2$ is kept **fixed** during optimization. Optimising it alongside the kernel hyperparameters tends to collapse it to zero, destabilising the Cholesky factorisation.